# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

`mlcroissant` allows us to inspect record sets, and, for each, detail fields and columns. All referencing uses the `@id` field. Below, we print all record set `@id`s and the associated fields.

In [ ]:
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record sets in this dataset.")
overview = {}
for rs in record_sets:
    print(f"\nRecordSet @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', '<no name>')}")
    print(f"  Description: {rs.get('description', '<no description>')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print(f"  Fields:")
    ids = []
    for f in fields:
        if isinstance(f, dict):
            field_id = f.get('@id', f)
            field_name = f.get('name', '<no name>')
        else:
            field_id = f
            field_name = '<unknown>'
        print(f"    - {field_id} ({field_name})")
        ids.append(field_id)
    overview[rs['@id']] = ids

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

For the purposes of this notebook, we'll extract data from the **main protein abundance record set**. Based on schema inspection, this is likely:

- Record Set: `cr:RecordSet.0` (replace with the actual @id after overview if different)

In [ ]:
# Adapt these values based on the overview printed above

# Example: Use the first record set available
if record_sets:
    main_record_set_id = record_sets[0]['@id']
    print(f"Using RecordSet @id: {main_record_set_id}")
else:
    raise ValueError("No record sets found in dataset.")

# Load records for each record set
rs_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in rs_ids:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records from {record_set_id}")
    except Exception as e:
        print(f"Skipped {record_set_id} due to error: {e}")

# Show column names for main record set
df_main = dataframes[main_record_set_id]
print(f"Columns in {main_record_set_id}:")
print(df_main.columns.tolist())
df_main.head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. We'll perform operations like removing outliers, transforming data distributions, and grouping data by attributes using field `@id`s.

Below we choose a likely quantitative field (e.g., total peptide count or abundance) for demonstration. Adjust the field referenced by `numeric_field_id` to your schema's actual `@id`/column name.

In [ ]:
# Select a numeric field for analysis
# Replace with correct field `@id` (or column name after inspection)

# Try to pick a numeric column
possible_numeric = [c for c in df_main.columns if ('count' in c or 'abundance' in c or 'MW' in c or 'coverage' in c or 'peptide' in c)]
# Fallback: just use the first column
if possible_numeric:
    numeric_field = possible_numeric[0]
else:
    numeric_field = df_main.columns[0]

print("Analyzing field:", numeric_field)

threshold = 10

# Try to convert column to numeric if it isn't already
df_main[numeric_field] = pd.to_numeric(df_main[numeric_field], errors='coerce')
filtered_df = df_main[df_main[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Find a group field (likely a categorical like 'sample' or 'accession')
possible_group = [c for c in df_main.columns if ('sample' in c or 'access' in c or 'group' in c)]
if possible_group:
    group_field = possible_group[0]
else:
    group_field = df_main.columns[1] if df_main.shape[1] > 1 else df_main.columns[0]

print(f"Grouping by: {group_field}")
if group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships. For instance, plot the distribution of the normalized numeric field, or a boxplot per group.

Below, we use `matplotlib` and `seaborn` (install if necessary) for plotting.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(8,4))
sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
plt.title(f'Normalized {numeric_field} Distribution')
plt.xlabel(f'{numeric_field} (normalized)')
plt.ylabel('Frequency')
plt.show()

# If group field is meaningful, plot group-wise boxplot
if group_field in filtered_df.columns and filtered_df[group_field].nunique() < 20:
    plt.figure(figsize=(10,4))
    sns.boxplot(x=group_field, y=f"{numeric_field}_normalized", data=filtered_df)
    plt.xticks(rotation=45)
    plt.title(f'{numeric_field} (normalized) by {group_field}')
    plt.show()

## 6. Conclusion
In this notebook, we:
- Loaded and inspected a Croissant-described mass spectrometry dataset.
- Explored available record sets and fields (referencing by `@id`).
- Extracted one record set as a DataFrame and performed basic EDA: filtering, normalization, and grouping.
- Visualized data distributions to spot potential trends or quality issues.

Further analyses could include deeper biological/clinical enrichment, multi-field correlations, or application-specific statistical modeling.